# Pandas + SQL practice

### Day 1: Skim and Review-- light practice

Want to review/ cover:
- Pandas: merge, groupby, pivot, apply
- SQL: SELECT, JOIN, GROUP BY, ORDER BY, WHERE and HAVING clauses
- Scikit-Learn: linear_model documentation

In [2]:
import pandas as pd

iris = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data', header = None)
iris.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']


In [3]:
# groupby
iris[iris['class'] == 'Iris-setosa'].groupby('class').describe()

'''
Summary:
- filtering rows is a bit harder-- have to filter it first and then do a group by-- different than R
- the '.' sortcut is much easier to use, but I wonder if there is a pipe sort of thing
- needs a function to group by-- like .mean(), .sum(), etc.
'''


"\nSummary:\n- filtering rows is a bit harder-- have to filter it first and then do a group by-- different than R\n- the '.' sortcut is much easier to use, but I wonder if there is a pipe sort of thing\n- needs a function to group by-- like .mean(), .sum(), etc.\n"

In [4]:
# merge
iris_a = iris[["class", "petal_width"]].groupby('class').mean()
iris_b = iris [['class', 'petal_length']].groupby('class').mean()

pd.DataFrame.merge(iris_a, iris_b, how = 'left', on= 'class')

'''
Summary:
- needs a function to group by-- like .mean(), .sum(), etc.
- double brackets filter cols, or can use .filter() function
- use of "how" and "on" to direct type of merge
'''

'\nSummary:\n- needs a function to group by-- like .mean(), .sum(), etc.\n- double brackets filter cols, or can use .filter() function\n- use of "how" and "on" to direct type of merge\n'

In [ ]:
# pivot
pd.DataFrame.pivot(iris, columns= 'class')
'''
Summary:
- use only when columns contain values and rows are the variables... in this case, this is incorrect use...
'''

''

In [18]:
#apply
import numpy as np
iris.apply(lambda x: x*2)
iris.apply(np.sum)

'''
Summary:
- pretty finicky-- two types of functions? I guess. one where it changes the data inside-- you have to refer
to current value as lamda x: and then have function like above. you can also do np.sum without parenthesis
in place of this
'''


'\nSummary:\n- pretty finicky-- two types of functions? I guess. one where it changes the data inside-- you have to refer\nto current value as lamda x: and then have function like above. you can also do np.sum without parenthesis\nin place of this\n'

### Day 2: SQL and Pandas Practice

##### SQL Queries

In [37]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("practice.db")

df = pd.DataFrame(iris)
df.to_sql("iris", conn, if_exists="replace", index=False)

query = """
WITH iris_a AS (
    SELECT class, AVG(petal_length) AS average_length
    FROM iris
    GROUP BY class
),
iris_counts AS (
    SELECT class, COUNT(*) AS number
    FROM iris
    WHERE class IN ('Iris-setosa', 'Iris-virginica')
    GROUP BY class
)
SELECT ic.class, ic.number, ia.average_length
FROM iris_counts ic
LEFT JOIN iris_a ia ON ic.class = ia.class
ORDER BY ic.number;
"""
result = pd.read_sql_query(query, conn)
print(result)

''' 
Summary:
- Lot of nitpicky things, but main thing is the order of queries: Where before groupby, order by is last, having after groupby
- cannot aggregate first and then join-- must do that in two CTEs
'''

            class  number  average_length
0     Iris-setosa      50           1.464
1  Iris-virginica      50           5.552


' \nSummary:\n- Lot of nitpicky things, but main thing is the order of queries: Where before groupby, order by is last, having after groupby\n- cannot aggregate first and then join-- must do that in two CTEs\n'

In [ ]:
# Window functions practice
conn = sqlite3.connect("practice.db")

df = pd.DataFrame(iris)
df.to_sql("iris", conn, if_exists="replace", index=False)

query = """
WITH iris_a AS (
    SELECT class, AVG(petal_length) AS average_length
    FROM iris
    GROUP BY class
),
iris_counts AS (
    SELECT class, COUNT(*) AS number
    FROM iris
    WHERE class IN ('Iris-setosa', 'Iris-virginica')
    GROUP BY class
)
SELECT ic.class, ic.number, ia.average_length, RANK() OVER (ORDER BY average_length DESC) AS rank
FROM iris_counts ic
LEFT JOIN iris_a ia ON ic.class = ia.class
ORDER BY ic.number;
"""
result = pd.read_sql_query(query, conn)
print(result)

''' 
Summary: 
- ROW_NUMBER vs RANK vs DENSE_RANK

'''

            class  number  average_length  rank
0  Iris-virginica      50           5.552     1
1     Iris-setosa      50           1.464     2


##### Actual SQL tasks
Compute average petal_length and petal_width for each class.

Find max and min sepal_length per class.

Count how many flowers are in each class.

In [40]:
df = pd.DataFrame(iris)
df.to_sql("iris", conn, if_exists="replace", index=False)

query = """
SELECT class, AVG(petal_length) as average_plength, AVG(petal_width) as average_pwidth,
MAX(sepal_length) as max_slength, MIN(sepal_length) AS min_slength, COUNT(*) as flowers
FROM iris
GROUP BY class;
"""
result = pd.read_sql_query(query, conn)
print(result)

             class  average_plength  average_pwidth  max_slength  min_slength  \
0      Iris-setosa            1.464           0.244          5.8          4.3   
1  Iris-versicolor            4.260           1.326          7.0          4.9   
2   Iris-virginica            5.552           2.026          7.9          4.9   

   flowers  
0       50  
1       50  
2       50  


Count flowers per class only for classes with more than 40 rows.

Find the classes with average petal_length > 4.5.

In [42]:
query1 = """
SELECT class, COUNT(*) as number
FROM iris
GROUP BY class
HAVING COUNT(*) > 40;
"""
query2 = """ 
SELECT class, AVG(petal_length) as average_length
FROM iris
GROUP BY class
HAVING average_length > 4.5;

"""
result = pd.read_sql_query(query2, conn)
print(result)

            class  average_length
0  Iris-virginica           5.552


Create a CTE with average petal_length per class, then join it back to the main table so each row knows its class average.

Join a table that contains max sepal_length per class to the main table.

In [44]:

query = """ 
WITH iris_a AS (
SELECT class, AVG(petal_length) as average_plength
FROM iris
GROUP BY class
),
iris_b AS (
SELECT class, MAX(sepal_length) as max_slength
FROM iris
GROUP BY class
)
SELECT iris.class, max_slength, average_plength
FROM iris
LEFT JOIN iris_a on iris.class = iris_a.class
LEFT JOIN iris_b on iris.class = iris_b.class;

"""
result = pd.read_sql_query(query, conn)
print(result)

              class  max_slength  average_plength
0       Iris-setosa          5.8            1.464
1       Iris-setosa          5.8            1.464
2       Iris-setosa          5.8            1.464
3       Iris-setosa          5.8            1.464
4       Iris-setosa          5.8            1.464
..              ...          ...              ...
145  Iris-virginica          7.9            5.552
146  Iris-virginica          7.9            5.552
147  Iris-virginica          7.9            5.552
148  Iris-virginica          7.9            5.552
149  Iris-virginica          7.9            5.552

[150 rows x 3 columns]


Rank flowers within each class by petal_length (use ROW_NUMBER()). -- Q1

Find the top 2 flowers by petal_length per class. --Q1

Compute the cumulative sum of petal_length within each class.--Q2

Use LAG() to compare each flower’s petal_length to the previous flower in the same class. --Q3

In [64]:
query1 = """ 
WITH rankings AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY class ORDER BY petal_length DESC) AS rank
FROM iris)
SELECT *
FROM rankings
WHERE rank <= 2
;
"""
query2 = """ 
WITH ordered AS (SELECT *, ROW_NUMBER() OVER (ORDER BY sepal_length DESC) AS row_id FROM iris) 
SELECT class, petal_length, SUM(petal_length) OVER (PARTITION BY class ORDER BY row_id) AS cumsum
FROM ordered;
"""

query3 = ''' 
WITH ordered AS (
SELECT *, ROW_NUMBER() OVER (ORDER BY sepal_length DESC) AS row_id
FROM iris
)
SELECT row_id, petal_length, LAG(petal_length) OVER (PARTITION BY class ORDER BY row_id)
FROM ordered
;
'''
result = pd.read_sql_query(query3, conn)
print(result)

     row_id  petal_length  \
0        71           1.2   
1        78           1.5   
2        79           1.7   
3        92           1.4   
4        93           1.3   
..      ...           ...   
145      76           5.1   
146      77           5.1   
147      85           5.0   
148      91           4.9   
149     134           4.5   

     LAG(petal_length) OVER (PARTITION BY class ORDER BY row_id)  
0                                                  NaN            
1                                                  1.2            
2                                                  1.5            
3                                                  1.7            
4                                                  1.4            
..                                                 ...            
145                                                5.1            
146                                                5.1            
147                                             

Divide flowers into 3 buckets per class based on petal_length

Compare average petal_length across buckets

In [66]:
query3 = ''' 
WITH bucketed AS (SELECT petal_length, NTILE(petal_length) OVER (PARTITION BY class) AS buckets
FROM iris)
SELECT buckets, AVG(petal_length) as average_plength
FROM bucketed
GROUP BY buckets
;
'''
result = pd.read_sql_query(query3, conn)
print(result)

   buckets  average_plength
0        1         2.502778
1        2         4.836364
2        3         4.910000
3        4         4.635000
4        5         5.462500
5        6         5.325000


Rank all flowers by petal_length across the dataset

Find the flower with the largest petal_length per class

In [69]:
query3 = ''' 
WITH ranked AS (
SELECT *, RANK() OVER (ORDER BY petal_length DESC) as ranking
FROM iris
)
SELECT class, MIN(ranking) AS max_plength
FROM ranked
GROUP BY class;
'''
result = pd.read_sql_query(query3, conn)
print(result)

             class  max_plength
0      Iris-setosa          101
1  Iris-versicolor           35
2   Iris-virginica            1


Identify flowers with petal_length above their class average

Label flowers as short / medium / long using conditional logic

Combine multiple CTEs into a single final query

In [73]:
query = ''' 
WITH class_average AS (
SELECT class, AVG(petal_length) AS average
FROM iris
GROUP BY class
)
SELECT iris.class, iris.petal_length, class_average.average, CASE
WHEN iris.petal_length >= class_average.average THEN "ok"
ELSE "not ok"
END AS label
FROM iris
LEFT JOIN class_average ON iris.class = class_average.class; 

'''
result = pd.read_sql_query(query, conn)
print(result)

              class  petal_length  average   label
0       Iris-setosa           1.4    1.464  not ok
1       Iris-setosa           1.4    1.464  not ok
2       Iris-setosa           1.3    1.464  not ok
3       Iris-setosa           1.5    1.464      ok
4       Iris-setosa           1.4    1.464  not ok
..              ...           ...      ...     ...
145  Iris-virginica           5.2    5.552  not ok
146  Iris-virginica           5.0    5.552  not ok
147  Iris-virginica           5.2    5.552  not ok
148  Iris-virginica           5.4    5.552  not ok
149  Iris-virginica           5.1    5.552  not ok

[150 rows x 4 columns]


### DAY 3: PANDAS FOLLOWUP

##### PANDAS version of this

In [74]:
import pandas as pd
iris = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/iris/iris.data', header = None)
iris.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 'class']

##### Actual Tasks and SQL equivalent:

SELECT: Select only the sepal-related columns

Select only the petal-related columns

Select all numeric columns

Rename one column to a shorter name

In [87]:
iris.filter(like = "sepal")
iris.filter(like = "petal")
iris.select_dtypes('number')
iris.rename(columns={"sepal_length" : "SL"})

,SL,sepal_width,petal_length,petal_width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica


2️⃣ Row Filtering (SQL: WHERE)

Filter rows where sepal length is greater than 5

Filter rows for one specific species

Filter rows where:

sepal length is greater than 5 and

petal length is greater than 1.5

Filter rows where:

sepal width is less than the dataset average

In [133]:
import numpy as np
iris.where(iris.sepal_length>5)
iris[iris["class"] == "Iris-setosa"]
iris[(iris["sepal_length"] > 5) & (iris["petal_length"] > 1.5)]

iris[iris["sepal_width"] < iris["sepal_width"].mean()]


,sepal_length,sepal_width,petal_length,petal_width,class
1,4.9,3.0,1.4,0.2,Iris-setosa
8,4.4,2.9,1.4,0.2,Iris-setosa
12,4.8,3.0,1.4,0.1,Iris-setosa
13,4.3,3.0,1.1,0.1,Iris-setosa
25,5.0,3.0,1.6,0.2,Iris-setosa
...,...,...,...,...,...
142,5.8,2.7,5.1,1.9,Iris-virginica
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica


3️⃣ Sorting (SQL: ORDER BY)

Sort all rows by sepal length (ascending)

Sort all rows by petal width (descending)

Sort species by their average petal length


In [ ]:
iris.sort_values(by = "sepal_length",ascending= 1)
iris.sort_values(by = "petal_width", ascending=0)
iris.groupby(by = "class").mean().sort_values(by = "petal_length")
''' 
Summary/ New thing learned: the . is kind of like the pipe in pandas
'''

,sepal_length,sepal_width,petal_length,petal_width,class
50,7.0,3.2,4.7,1.4,Iris-versicolor
51,6.4,3.2,4.5,1.5,Iris-versicolor
52,6.9,3.1,4.9,1.5,Iris-versicolor
53,5.5,2.3,4.0,1.3,Iris-versicolor
54,6.5,2.8,4.6,1.5,Iris-versicolor
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,Iris-virginica
146,6.3,2.5,5.0,1.9,Iris-virginica
147,6.5,3.0,5.2,2.0,Iris-virginica
148,6.2,3.4,5.4,2.3,Iris-virginica



4️⃣ Grouping & Aggregation (SQL: GROUP BY)

Group by species and compute:

mean sepal length

Group by species and compute:

mean and median petal length

Count the number of observations per species

Identify which species has the largest average sepal width


In [150]:
iris.groupby("class").median()
iris.groupby("class").count()
iris.groupby("class").max("sepal_width")


,sepal_length,sepal_width,petal_length,petal_width
class,,,,
Iris-setosa,5.8,4.4,1.9,0.6
Iris-versicolor,7.0,3.4,5.1,1.8
Iris-virginica,7.9,3.8,6.9,2.5



5️⃣ Limiting Results (SQL: LIMIT)

Return the top 5 rows with the largest petal length

Return the bottom 5 rows with the smallest sepal length

Return the top species by average petal width

In [ ]:
iris.sort_values(by = "petal_length", ascending=0).head(5)
iris.sort_values(by = "sepal_length", ascending=0).tail(5)
iris.groupby("class").mean("petal_width").sort_values(by = "petal_width", ascending=0).iloc[[0]]

''' 
Things to note: double brackets pull out dataframe, single pulls out series
'''

,sepal_length,sepal_width,petal_length,petal_width
class,,,,
Iris-virginica,6.588,2.974,5.552,2.026


6️⃣ Derived / Calculated Columns

Create a new column representing sepal area

Create a new column representing petal area

Compute the average sepal area for each species

In [180]:
iris["sepal_area"] = iris.sepal_length * iris.sepal_width
iris["petal_area"] = iris.petal_length *iris.petal_width
iris.groupby("class").mean()[["sepal_area"]]


,sepal_area
class,
Iris-setosa,17.2088
Iris-versicolor,16.5262
Iris-virginica,19.6846


7️⃣ One Full Logic Prompt (SQL-style Thinking)

Prompt:

For each species, compute the average petal length and petal width, then return the species ordered by average petal length from largest to smallest.

In [181]:
iris.groupby("class").mean()[["petal_length", "petal_width"]].sort_values(by = "petal_length", ascending=0)

,petal_length,petal_width
class,,
Iris-virginica,5.552,2.026
Iris-versicolor,4.260,1.326
Iris-setosa,1.464,0.244


# Feb 11, 2026

In [13]:
import sqlite3
import pandas as pd
conn = sqlite3.connect("practice.db")
titanic = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
practice_df = pd.DataFrame(titanic)
practice_df.to_sql("titanic", conn, if_exists="replace", index=False)

query = '''
WITH age AS (
SELECT Sex, AVG(Age) AS average_age
FROM titanic
GROUP BY Sex
),
fare AS (
SELECT Sex, AVG(Fare) AS average_fare
FROM titanic
GROUP BY Sex
)
SELECT fare.Sex, average_fare, average_age
FROM fare
INNER JOIN age ON fare.Sex = age.Sex;
'''
result = pd.read_sql_query(query, conn)
print(result)

      Sex  average_fare  average_age
0  female     44.479818    27.915709
1    male     25.523893    30.726645


In [36]:
query = '''
SELECT PassengerID, Pclass, Fare, RANK() OVER(PARTITION BY Pclass ORDER BY Fare DESC) AS Rank
FROM titanic;
'''
result = pd.read_sql_query(query, conn)
print(result)

     PassengerId  Pclass      Fare  Rank
0            259       1  512.3292     1
1            680       1  512.3292     1
2            738       1  512.3292     1
3             28       1  263.0000     4
4             89       1  263.0000     4
..           ...     ...       ...   ...
886          379       3    4.0125   487
887          180       3    0.0000   488
888          272       3    0.0000   488
889          303       3    0.0000   488
890          598       3    0.0000   488

[891 rows x 4 columns]


Use RANK() if ties should share rank

Use ROW_NUMBER() if you need exactly one top row

Use DENSE_RANK() if ranking categories without gaps